# Motor STT con faster-whisper

Transcripción de audio a texto usando **faster-whisper** acelerado por GPU (RTX 3050).

## 1. Verificación de GPU

Confirmamos que CTranslate2 (el backend de faster-whisper) ve la GPU CUDA.

In [ ]:
import ctranslate2

n_gpu = ctranslate2.get_cuda_device_count()
print(f'Dispositivos CUDA detectados: {n_gpu}')
print('GPU disponible' if n_gpu > 0 else '¡Sin GPU! Ajustar device a "cpu"')

## 2. Carga del modelo

Modelos: `tiny`, `base`, `small`, `medium`, `large-v3`. Con 4GB de VRAM recomiendo `small` o `base`.

- `device='cuda'`: acelera con la RTX 3050.
- `compute_type='float16'`: usa la mitad de VRAM manteniendo buena precisión.

In [ ]:
from faster_whisper import WhisperModel

MODELO = 'small'          # tiny | base | small | medium | large-v3
model = WhisperModel(MODELO, device='cuda', compute_type='float16')
print(f'Modelo "{MODELO}" cargado en CUDA')

## 3. Transcribir un audio

Ajusta `AUDIO` a la ruta de tu archivo (`.mp3`, `.wav`, `.m4a`, etc.).

- `language='es'`: fija el idioma (opcional, sino lo detecta solo).
- `beam_size=5`: calidad de la búsqueda (más alto = mejor, más lento).

In [ ]:
from pathlib import Path

AUDIO = Path('audio/muestra.wav')
assert AUDIO.exists(), f'No existe el archivo: {AUDIO}'

segments, info = model.transcribe(
    str(AUDIO),
    language='es',
    beam_size=5,
)

print(f'Idioma detectado: {info.language} | Duración: {info.duration:.1f}s')

transcripcion = []
for seg in segments:
    transcripcion.append(seg.text.strip())
    print(f'[{seg.start:7.2f} -> {seg.end:7.2f}] {seg.text.strip()}')

## 4. Guardar la transcripción

In [ ]:
texto_completo = ' '.join(transcripcion)
out = Path('transcripcion.txt')
out.write_text(texto_completo, encoding='utf-8')
print(f'Guardado en {out.resolve()}')
texto_completo

## 5. Notas

- **Subir modelo**: de `small` a `medium`/`large-v3` para mayor precisión (más VRAM y más lento).
- **Sin GPU**: cambiar a `device='cpu', compute_type='int8'`.
- **Formato timestamp**: consultar la doc de faster-whisper para exportar `.srt`/`.vtt`.